Run the following cell only once: install nirfasterff and download the necessary sample data.
You may use the same code if you are running local. The correct version of nirfasterff should be automatically determined.

In [ ]:
import platform
import subprocess

version_string = platform.python_version()
major_ver = version_string.split('.')[1]
!git clone https://github.com/milabuob/nirfaster-FF.git
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/Conte69_on_TT_32k.mat
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/IM_Gordon_2014_333_Parcels.mat
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/headvol.mat
!wget -c https://github.com/JiamingCao/nirfast-atlas/raw/refs/heads/main/headvol_optodes.mat

current_os = platform.system()
if current_os == "Darwin":
    cpulib_name = 'cpu-mac-python3'+major_ver+'.zip'
else:
    if current_os == "Windows":
        cpulib_name = 'cpu-win-python3'+major_ver+'.zip'
        gpulib_name = 'gpu-win-python3'+major_ver+'.zip'
        subprocess.run(['wget', '-c', 'https://github.com/milabuob/nirfaster-FF/releases/download/v1.3.0/'+gpulib_name])
        subprocess.run(['unzip', gpulib_name, '-d', 'nirfaster-FF/nirfasterff/lib'])
        subprocess.run(['rm', gpulib_name])
    elif current_os == "Linux":
        cpulib_name = 'cpu-linux-python3'+major_ver+'.zip'
        gpulib_name = 'gpu-linux-python3'+major_ver+'.zip'
        subprocess.run(['wget', '-c', 'https://github.com/milabuob/nirfaster-FF/releases/download/v1.3.0/'+gpulib_name])
        subprocess.run(['unzip', gpulib_name, '-d', 'nirfaster-FF/nirfasterff/lib'])
        subprocess.run(['rm', gpulib_name])
        
subprocess.run(['wget', '-c', 'https://github.com/milabuob/nirfaster-FF/releases/download/v1.3.0/'+cpulib_name])
subprocess.run(['unzip', cpulib_name, '-d', 'nirfaster-FF/nirfasterff/lib'])
subprocess.run(['rm', cpulib_name])

# comment out the following lines if you run local
!pip install ipympl
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
import sys
import os
sys.path.insert(1, 'nirfaster-FF/')
import nirfasterff as ff # ff is short for fast and furious
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import copy
import math
# enable interactive matplotlib
# comment out the following lines if you run local
from google.colab import output
output.enable_custom_widget_manager()
%matplotlib ipympl

**IMPORTANT:** make sure you have uploaded your previously created "meshes.zip" to the Colab session. Unzip the mesh files.

In [ ]:
!unzip -o meshes.zip

In [ ]:
# load the mesh we created in a different notebook
mesh_760nm = ff.base.stndmesh()
mesh_760nm.from_file('meshes/headmesh_760nm')
# make sure the sources are 1 scattering distance inside external boundary, and detectors are on the boundary
# If you created the mesh using our tutorial, the optodes were already moved to the correct locations upon mesh creation, 
# and this step is redundant
# Run the following line if you are unsure
mesh_760nm.touch_optodes()

mesh_850nm = ff.base.stndmesh()
mesh_850nm.from_file('meshes/headmesh_850nm')
# make sure the sources are 1 scattering distance inside external boundary, and detectors are on the boundary
# If you created the mesh using our tutorial, the optodes were already moved to the correct locations upon mesh creation, 
# and this step is redundant
# Run the following line if you are unsure
mesh_850nm.touch_optodes()

### Part 1: Generating forward data using a realistic head model

In [ ]:
# View the external surface of the mesh, and the source/detector location
bndvtx = np.where(mesh_760nm.bndvtx==1)

%matplotlib widget
fig = plt.figure()
ax = plt.axes(projection='3d')

ax.plot3D(mesh_760nm.nodes[bndvtx,0],mesh_760nm.nodes[bndvtx,1],mesh_760nm.nodes[bndvtx,2],'.')

# create some little spheres for better visibility: matplotlib does not have true 3D markers
u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
xs = np.cos(u) * np.sin(v)
ys = np.sin(u) * np.sin(v)
zs = np.cos(v)
ri = 2.5

for xi, yi, zi in mesh_760nm.source.coord:
     ax.plot_surface(xi + ri * xs, yi + ri * ys, zi + ri * zs, color='red', edgecolor='None')

ri = 3
for xi, yi, zi in mesh_760nm.meas.coord:
    ax.plot_surface(xi + ri * xs, yi + ri * ys, zi + ri * zs, color='black', edgecolor='None')

plt.axis('equal')
plt.show()

In [ ]:

# 1. Load datasets for parcels
# The Cortical Area Parcellation from Resting-State Correlations dataset consists of 333 cortical patches segmented using resting-state fMRI (Gordon 2014).
gordon = sio.loadmat("IM_Gordon_2014_333_Parcels.mat")

# surface atlas constructed by David Van Essen's group, derived from structural MRI scans of 69 healthy control subjects. 
# It serves as a standard reference space for comparing cortical geometries across different people
# we use the nodal coordinates
conte = sio.loadmat("Conte69_on_TT_32k.mat") 

# 2. Get parcel ID. The 13 parcels available are:
# 'Auditory','CinguloOperc','CinguloParietal','Default','DorsalAttn','FrontoParietal','None','RetrosplenialTemporal','SMhand','SMmouth','Salience','VentralAttn','Visual'
#  choose 'left' or 'right' and set activation size (radius, in mm)
parcel = "Auditory"
side = "right"
size = 15

# get cortex nodal index of chosen activation
im = gordon["IM"][0, 0]
im_nets = im["Nets"].flatten()
keep_idx = np.where(im_nets == parcel)[0] + 1  # +1 matches MATLAB 1-based indexing

parcel_nets = gordon["Parcel_Nets"][0, 0]

# 3. Locate matching brain surface node coordinates
if side == "left":
    ctx_data = parcel_nets["CtxL"].flatten()
    ctx_nodes = conte["Anat"]["CtxL"][0, 0]["nodes"][0, 0]
elif side == "right":
    ctx_data = parcel_nets["CtxR"].flatten()
    ctx_nodes = conte["Anat"]["CtxR"][0, 0]["nodes"][0, 0]

matching_indices = np.where(np.isin(ctx_data, keep_idx))[0]

# now find nodes in gray matter in our mesh, that falls within this activation
gray = np.where(mesh_760nm.region==3)

# compute the final mean coordinate vector and get distance of all nodes from that point
result_mean = ctx_nodes[matching_indices, :].mean(axis=0)
result = np.linalg.norm(mesh_760nm.nodes-result_mean,axis=1)

# get parcel index
ind = np.where(result <= size)[0]
parcel_indx = np.intersect1d(gray, ind)

# plot the activation within our mesh
%matplotlib widget
fig = plt.figure()
ax = plt.axes(projection='3d')

ax.plot3D(mesh_760nm.nodes[bndvtx,0],mesh_760nm.nodes[bndvtx,1],mesh_760nm.nodes[bndvtx,2],'c.')
ax.plot3D(mesh_760nm.nodes[gray,0],mesh_760nm.nodes[gray,1],mesh_760nm.nodes[gray,2],'w.')
ax.plot3D(mesh_760nm.nodes[parcel_indx,0],mesh_760nm.nodes[parcel_indx,1],mesh_760nm.nodes[parcel_indx,2],'k.')
ax.plot3D(result_mean[0],result_mean[1],result_mean[2],'rx')

ax.plot3D(mesh_760nm.source.coord[:,0],mesh_760nm.source.coord[:,1],mesh_760nm.source.coord[:,2],'ro')

In [ ]:
# we need to do the same in voxel space. This is because we calculate the Jacobian in voxel space.
# Suppose we use a 5mm resolution grid,
mesh_760nm.voxelize(5)
mesh_850nm.voxelize(5)
# get the coordinates of voxels, so we can identify the correct voxels for activation
X, Y, Z = np.meshgrid(mesh_760nm.vol.xgrid, mesh_760nm.vol.ygrid, mesh_760nm.vol.zgrid)
coords = np.c_[X.flatten('F'), Y.flatten('F'), Z.flatten('F')]

# convert region label to voxel space
regions_vol = mesh_760nm.nntogrid('region').flatten('F')

# calculate the distance between the voxels and the center of the activation region
grayvol = np.where(regions_vol==3)
distvol = np.linalg.norm(coords - result_mean, axis=1)
indvol = np.where(distvol <= size)[0]
parcel_indxvol = np.intersect1d(grayvol, indvol)

# visualize
vol2 = np.zeros(regions_vol.shape)
vol2[grayvol] = 1
vol2[parcel_indxvol] = 2
mesh_760nm.plotvol(vol2, bnd=1)

In [ ]:
# define HRF function
from scipy.special import gamma
def canonicalHRF(duration, Fs, peakTime=6., uShootTime=16., peakDisp=1.,
                 uShootDisp=1., ratio=1/6):
    # canonical hrf function adapted from nirs-toolbox, using standard SPM parameters
    t = np.arange(0, duration, 1/Fs)
    a1 = peakTime
    a2 = uShootTime
    b1 = peakDisp
    b2 = uShootDisp
    c  = ratio
    h = (b1**a1) * t**(a1-1)*np.exp(-b1*t)/gamma(a1) - c*(b2**a2)*(t**(a2-1))*np.exp(-b2*t)/gamma(a2)
    return h/h.max()

In [ ]:
# create a Haemodynamic Response Function (HRF) for HbO and Hb
hrf_length = 30 # duration (sec)
Fs = 10.0 # sampling rate (hz)
hrf_HbO = canonicalHRF(hrf_length, Fs)

# assuming Hb HRF is -0.25 of HbO
hrf_Hb = -0.25 * hrf_HbO
tvec = np.arange(0, hrf_length, 1/Fs)

plt.close('all')
plt.plot(tvec, hrf_HbO, 'r', label='HbO')
plt.plot(tvec, hrf_Hb, label='Hb')
plt.legend()
plt.show()

In [ ]:
# Calculate change in mua from the HRF using Beer's Law
hrf = np.vstack((hrf_HbO,hrf_Hb))

# let's define an extinction coefficient matrix
E = np.array([[1.3495580e-01,   3.5662416e-01],
              [2.4365740e-01,   1.5921100e-01]])

delta_mua = np.dot(E,hrf)

plt.close('all')
plt.plot(tvec, delta_mua[0,:], 'r', label='760nm')
plt.plot(tvec, delta_mua[1,:], label='850nm')
plt.xlabel('time (s)')
plt.ylabel(r'$\delta \mu_a$')
plt.legend()
plt.show()

In [ ]:
# calculate the Jacobian matrix for each mesh, at each wavelength
J_760nm = mesh_760nm.jacobian(0)[0]
J_850nm = mesh_850nm.jacobian(0)[0]

In [ ]:
# visualize one of the bananas
mesh_760nm.plotvol(J_760nm[15,:], bnd=1)

In [ ]:
# visualize total sensitivity
mesh_760nm.plotvol(np.sum(J_760nm,axis=0), bnd=1)

In [ ]:
# perturbation, but only the voxels within the focal activation
# note that we normally only need about 5% change in data
nchannels, nvoxels = J_760nm.shape
# 760 nm
dmua = np.zeros((nvoxels, tvec.size))
dmua[parcel_indxvol,:] = delta_mua[0,:]
# dmua[:,:] = delta_mua[0,:]
dOD_760nm = - J_760nm @ (dmua)

# 850 nm
dmua = np.zeros((nvoxels, tvec.size))
dmua[parcel_indxvol,:] = delta_mua[1,:]
# dmua[:,:] = delta_mua[0,:]
dOD_850nm = - J_850nm @ (dmua)

plt.close('all')
plt.subplot(2,1,1)
plt.plot(tvec, dOD_760nm.T)
plt.title('760 nm')
plt.subplot(2,1,2)
plt.plot(tvec, dOD_850nm.T)
plt.title('850 nm')
plt.tight_layout()
plt.show()


In [ ]:
# let's apply the Modified Beer-Lambert solution here for a channel with highest OD
channel = np.where(np.abs(dOD_760nm)==np.max(np.abs(dOD_760nm)))[0]
print(channel)
print(mesh_760nm.link[channel,:][0])

DPF = 6

# get channel distance
d = np.sqrt(np.sum((mesh_760nm.source.coord[21,:]-mesh_760nm.meas.coord[11,:]) ** 2))

dmua = np.zeros((2,300))
dmua[0,:] = dOD_760nm[channel,:] / (DPF * d) 
dmua[1,:] = dOD_850nm[channel,:] / (DPF * d)
result = np.linalg.inv(E) @ dmua

# for plotting purpose, normalise and compare to original
hrf_recoeverd = result / np.max(np.abs(result))

plt.close('all')
plt.plot(tvec, hrf_recoeverd[0,:], 'r', label='HbO')
plt.plot(tvec, hrf_recoeverd[1,:], 'b', label='Hb')
plt.plot(tvec, hrf_HbO, 'rx', label='HbO: target')
plt.plot(tvec, hrf_Hb, 'b.', label='Hb')
plt.legend()
plt.show()

### Part 2: Basic reconstruction, and adding realitic noise to data

In [ ]:
# a very simple reconstruction at the peak, NeuroDOT-style regularization
alpha = 0.01
beta = 0.01
L_760nm = np.sqrt(alpha*(np.sum(J_760nm**2, axis=0)+beta))
L_850nm = np.sqrt(alpha*(np.sum(J_850nm**2, axis=0)+beta))

recon_760nm = ff.inverse.tikhonov(J_760nm, L_760nm, - dOD_760nm[:,50])
recon_850nm = ff.inverse.tikhonov(J_850nm, L_850nm, - dOD_850nm[:,50])
recon = np.vstack((recon_760nm,recon_850nm))
result = np.linalg.inv(E) @ recon

reconvol_HbO = np.reshape(result[0,:], (mesh_760nm.vol.xgrid.size, mesh_760nm.vol.ygrid.size, mesh_760nm.vol.zgrid.size), order='F')
reconvol_Hb = np.reshape(result[1,:], (mesh_760nm.vol.xgrid.size, mesh_760nm.vol.ygrid.size, mesh_760nm.vol.zgrid.size), order='F')


mesh_760nm.plotvol(reconvol_HbO, bnd=1, cmap = 'jet',surfcnt=50, clim=[- np.max(np.abs(reconvol_HbO)), np.max(np.abs(reconvol_HbO))])
mesh_760nm.plotvol(reconvol_Hb, bnd=1, cmap = 'jet',surfcnt=50, clim=[- np.max(np.abs(reconvol_Hb)), np.max(np.abs(reconvol_Hb))])


In [ ]:
# first calculate the channel lengths
nchannels = mesh_760nm.link.shape[0]
SDS = np.zeros(nchannels)
for i in range(nchannels):
    SDS[i] = np.linalg.norm(mesh_760nm.source.coord[mesh_760nm.link[i,0]-1,:] - mesh_760nm.meas.coord[mesh_760nm.link[i,1]-1,:])

In [ ]:
# noisy data after perturbation
# noise model is the same as the semi-infinite model, taken from paper doi: 10.1117/1.NPh.6.3.035007
noise_percent850 = 0.6019*np.exp(0.01052*SDS) + 9.685e-5*np.exp(0.1382*SDS)
noise_percent760 = 0.2502*np.exp(0.02913*SDS) + 4.625e-6*np.exp(0.2128*SDS)

noise850 = np.random.randn(dOD_850nm.shape[0], dOD_850nm.shape[1]) * (np.max(dOD_850nm) * noise_percent850[:,None]/100)
noise760 = np.random.randn(dOD_850nm.shape[0], dOD_850nm.shape[1]) * (np.max(dOD_760nm) * noise_percent760[:,None]/100)

dOD_850nm_noisy = dOD_850nm + noise850
dOD_760nm_noisy = dOD_760nm + noise760

In [ ]:
plt.close('all')
plt.subplot(2,1,1)
plt.plot(tvec, dOD1_760nm_noisy.T)
plt.title('760 nm')
plt.subplot(2,1,2)
plt.plot(tvec, dOD1_850nm_noisy.T)
plt.title('850 nm')
plt.tight_layout()
plt.show()

**Homework:** If you run this cell, you will notice that the reconstruction result now looks awful because of the noise. In this case, it is usually a good idea to adjust the value of alpha according to how much noise the data contains. Have a play with it, and see if you can make the reconstruction results look nice again.

In [ ]:
# a very simple reconstruction at the peak, NeuroDOT-style regularization
alpha = 0.01 
beta = 0.01
L_760nm = np.sqrt(alpha*(np.sum(J_760nm**2, axis=0)+beta))
L_850nm = np.sqrt(alpha*(np.sum(J_850nm**2, axis=0)+beta))

recon_760nm = ff.inverse.tikhonov(J_760nm, L_760nm, -dOD1_760nm_noisy[:,50])
recon_850nm = ff.inverse.tikhonov(J_850nm, L_850nm, -dOD1_850nm_noisy[:,50])
recon = np.vstack((recon_760nm,recon_850nm))
result = np.linalg.inv(E) @ recon

reconvol_HbO = np.reshape(result[0,:], (mesh_760nm.vol.xgrid.size, mesh_760nm.vol.ygrid.size, mesh_760nm.vol.zgrid.size), order='F')
reconvol_Hb = np.reshape(result[1,:], (mesh_760nm.vol.xgrid.size, mesh_760nm.vol.ygrid.size, mesh_760nm.vol.zgrid.size), order='F')


mesh_760nm.plotvol(reconvol_HbO, bnd=1, cmap = 'jet',surfcnt=50, clim=[- np.max(np.abs(reconvol_HbO)), np.max(np.abs(reconvol_HbO))])
mesh_760nm.plotvol(reconvol_Hb, bnd=1, cmap = 'jet',surfcnt=50, clim=[- np.max(np.abs(reconvol_Hb)), np.max(np.abs(reconvol_Hb))])